# EDA inicial de archivos crudos

Este notebook revisa los CSV generados desde `data/raw/csv`, detecta la fila de encabezado real y construye un perfil básico por archivo.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/csv")
CSV_FILES = sorted(DATA_DIR.glob("*.csv"))

if not CSV_FILES:
    raise FileNotFoundError(f"No se encontraron CSV en {DATA_DIR.resolve()}")

CSV_FILES

In [ ]:
def detect_header_row(csv_path: Path, search_rows: int = 10) -> int:
    preview = pd.read_csv(csv_path, header=None, nrows=search_rows, dtype=str)
    for row_index, row in preview.iterrows():
        values = [str(value).strip() for value in row.fillna("")]
        if any("Identificación de muestra" in value for value in values):
            return int(row_index)
    raise ValueError(f"No se detectó encabezado en {csv_path.name}")


def load_structured_csv(csv_path: Path) -> tuple[pd.DataFrame, int]:
    header_row = detect_header_row(csv_path)
    df = pd.read_csv(csv_path, header=header_row, dtype=str)
    df.columns = [str(column).strip() for column in df.columns]
    df = df.dropna(how="all")
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    return df, header_row


def build_profile(csv_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    df, header_row = load_structured_csv(csv_path)
    nd_counts = (df == "nd").sum()
    blank_counts = df.isna().sum() + (df == "").sum()

    summary = pd.DataFrame(
        {
            "archivo": [csv_path.name],
            "header_row": [header_row],
            "filas": [len(df)],
            "columnas": [len(df.columns)],
            "columnas_con_nd": [int((nd_counts > 0).sum())],
            "total_nd": [int(nd_counts.sum())],
        }
    )

    column_profile = pd.DataFrame(
        {
            "archivo": csv_path.name,
            "columna": df.columns,
            "no_nulos": df.notna().sum().values,
            "vacios": blank_counts.reindex(df.columns).values,
            "nd": nd_counts.reindex(df.columns).values,
            "muestra": [df[column].dropna().head(3).tolist() for column in df.columns],
        }
    )
    return summary, column_profile


In [ ]:
summaries = []
column_profiles = []

for csv_path in CSV_FILES:
    summary, column_profile = build_profile(csv_path)
    summaries.append(summary)
    column_profiles.append(column_profile)

summary_df = pd.concat(summaries, ignore_index=True)
columns_df = pd.concat(column_profiles, ignore_index=True)

summary_df.sort_values("archivo")

In [ ]:
columns_df.sort_values(["archivo", "columna"])

In [ ]:
example_file = CSV_FILES[0]
example_df, example_header_row = load_structured_csv(example_file)

print(f"Archivo ejemplo: {example_file.name}")
print(f"Fila detectada como encabezado: {example_header_row}")
example_df.head(10)

In [ ]:
numeric_preview = example_df.replace("nd", pd.NA)
numeric_columns = [column for column in numeric_preview.columns if "ppm" in column]
numeric_preview[numeric_columns] = numeric_preview[numeric_columns].apply(pd.to_numeric, errors="coerce")
numeric_preview[numeric_columns].describe().T